# Supported-DPSS RFI flagger — v3-beta review

This notebook is a thin review interface to `eigsep_data.rfi_supported`. It contains no legacy comparison or artificial injection. It selects data, applies default parameters plus explicit overrides, runs the flagger, and presents the five requested aligned views.

**Status:** `v3-beta`; the method and product schemas remain subject to review.

In [ ]:
from pathlib import Path
import sys
from dataclasses import asdict, replace
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, SymLogNorm
from IPython.display import display

%matplotlib widget

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'eigsep_data/src/eigsep_data').is_dir())
sys.path.insert(0, str(ROOT / 'eigsep_data/src'))

from eigsep_data import MetadataIndex
from eigsep_data.rfi_supported import FLAG_BITS, REPORT_BANDS, RFIConfig, run_selection

CAMPAIGN = ROOT / 'marjum-2026-07'
print('Campaign:', CAMPAIGN)
print('Algorithm: flags@v3-beta.1 / smooth_model@v3-beta.1')

## 1. Select input data

Set `FILES` to an inclusive pair of complete filenames, or set `TIME` to a `(start, end)` UTC/Unix pair. With both left as `None`, the default selects a 20-minute window ten hours after the start of visit 13. Physical antenna names are resolved against every HDF5 file; no correlator key is assumed to retain one meaning across the selection.

In [ ]:
FILES = None
TIME = None

SPAN = 14
OFFSET_S = 1 * 3600
MINUTES = 20

AIR_ANTENNA = 'box-air'
GROUND_ANTENNA = 'box-gnd'

idx = MetadataIndex(CAMPAIGN / 'data')
if FILES is not None:
    selection = idx.select(files=tuple(FILES))
elif TIME is not None:
    selection = idx.select(time=tuple(TIME))
else:
    all_meta = idx.select().meta.assign(visit=idx.select().visits(gap_s=600))
    visit = all_meta[all_meta.visit == SPAN]
    start = float(visit.time_best.min()) + OFFSET_S
    selection = idx.select(time=(start, start + MINUTES * 60))

display(selection.meta.groupby(['file', 'rfswitch'], dropna=False).size().rename('rows'))
print(selection.summary())

def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

display(pd.Series({name: sha256(CAMPAIGN / 'data' / name)
                   for name in selection.files}, name='input sha256'))
print('Algorithm source sha256:', sha256(ROOT / 'eigsep_data/src/eigsep_data/rfi_supported.py'))

## 2. Parameters

Edit only `OVERRIDES`. Empty overrides use the package defaults recorded in the product manifest. The background combines the full-band 50 ns by 1 mHz DPSS continuum basis with a 300 ns basis over 35–88 MHz whose amplitudes are constant over this window. The low-band correction is tapered over 5 MHz at both edges and fitted jointly with the continuum. Set `spectral_correction_halfwidth_s` to zero for the previous model.

Units are seconds for frequency half-widths, Hz for `time_halfwidth_hz`, MHz for frequency bands and taper width, and integrations/channels for guards.

In [ ]:
OVERRIDES = {
    # 'cross_cut': 8.0,
    # 'support_inflation': 5.0,
}

config = replace(RFIConfig(), **OVERRIDES)
display(pd.Series(asdict(config), name='v3-beta parameters'))

## 3. Generate the mask and supported model

Non-`RFANT` rows are excluded before fitting and receive bit 0 over the full band. The original continuum and constant-time low-band correction are solved together, and the design-support calculation covers their combined coefficient set. The public model remains NaN where those support checks fail; bit 7 records those cells. The in-memory `model_raw` is retained only for diagnosis.

In [ ]:
result = run_selection(
    selection,
    air_antenna=AIR_ANTENNA,
    ground_antenna=GROUND_ANTENNA,
    config=config,
)

air = selection.load_bundle(antenna=AIR_ANTENNA, band_mhz=config.band_mhz,
                            missing='raise')
residual = air.data - result.model
flagged_residual = np.where(result.mask, np.nan, residual)

display(pd.DataFrame(result.diagnostics['resolved_inputs']).T.rename_axis('file'))

summary = pd.Series({
    'rows': len(result.times),
    'channels': len(result.freqs_mhz),
    'excluded_fraction': result.mask.mean(),
    'rfi_evidence_fraction': result.rfi_mask.mean(),
    'supported_model_fraction': result.support_ok.mean(),
    'fit_training_fraction': result.fit_keep.mean(),
}, name='result')
display(summary)
first_segment = result.diagnostics['segments'][0]
display(pd.Series({
    'coefficients': first_segment['solver'][-1]['coefficients'],
    'constant_time_spectral_modes': first_segment['solver'][-1]['stationary_frequency_modes'],
}, name='combined model design'))
display(pd.Series({name: mask.mean() for name, mask in result.reasons.items()},
                  name='overlapping reason fractions'))
display(pd.DataFrame(FLAG_BITS).T)

band_rows = []
for name, (lo, hi) in {'35-50': (35, 50), '50-70': (50, 70), **REPORT_BANDS}.items():
    channel = (result.freqs_mhz >= lo) & (result.freqs_mhz < hi)
    band_rows.append({
        'band_mhz': name,
        'supported': result.support_ok[:, channel].mean(),
        'fit_training': result.fit_keep[:, channel].mean(),
        'retained': (~result.mask[:, channel]).mean(),
        'rfi_evidence': result.rfi_mask[:, channel].mean(),
    })
display(pd.DataFrame(band_rows).set_index('band_mhz'))

## 4. Shared data/model/flag/residual view

All five panels share frequency and time axes and remain interactive under `%matplotlib widget`. Residual panels use raw correlator counts. White gaps in the model are deliberately unsupported, and white gaps in the last panel are all excluded cells.

In [ ]:
freqs = result.freqs_mhz
minutes = (result.times - result.times[0]) / 60
extent = [freqs[0], freqs[-1], minutes[-1], 0]

positive = air.data[np.isfinite(air.data) & (air.data > 0)]
vmin, vmax = np.quantile(positive, [0.01, 0.99])
finite_residual = residual[np.isfinite(residual)]
rlim = np.quantile(np.abs(finite_residual), 0.9)
linthresh = max(1.0, np.quantile(np.abs(finite_residual), 0.5))

fig, axes = plt.subplots(1, 5, figsize=(25, 8), sharex=True, sharey=True,
                         constrained_layout=True)
count_norm = LogNorm(vmin=max(vmin, 1), vmax=vmax)
residual_norm = SymLogNorm(linthresh=linthresh, vmin=-rlim, vmax=rlim)

images = []
images.append(axes[0].imshow(air.data, aspect='auto', extent=extent,
                             interpolation='nearest', norm=count_norm))
images.append(axes[1].imshow(result.model, aspect='auto', extent=extent,
                             interpolation='nearest', norm=count_norm))
images.append(axes[2].imshow(result.mask, aspect='auto', extent=extent,
                             interpolation='nearest', cmap='gray_r', vmin=0, vmax=1))
images.append(axes[3].imshow(residual, aspect='auto', extent=extent,
                             interpolation='nearest', cmap='RdBu_r', norm=residual_norm))
images.append(axes[4].imshow(flagged_residual, aspect='auto', extent=extent,
                             interpolation='nearest', cmap='RdBu_r', norm=residual_norm))

for axis, title in zip(axes, ['Data', 'Supported model', 'Flags', 'Residual', 'Flagged residual']):
    axis.set_title(title)
    axis.set_xlabel('Frequency [MHz]')
axes[0].set_ylabel('Minutes from selection start')
fig.colorbar(images[0], ax=axes[:2], label='Correlator counts', shrink=0.75)
fig.colorbar(images[3], ax=axes[3:], label='Data − model [counts]', shrink=0.75)
plt.show()

## Provenance and caveats

- Input time is `time_best` from `MetadataIndex`; physical antennas are resolved independently for every HDF5 file, and the resulting auto/cross keys and cross conjugation are shown above.
- The tested background-model domain is 35–235 MHz. Outside it, the public model is unsupported and the mask is set.
- The 300 ns correction has constant amplitude over this selected window. It can represent stable 35–88 MHz spectral structure, including stable interference; recovered pixels should not be interpreted as clean sky from the model alone.
- The correction is tapered at 35 and 88 MHz. The combined support diagnostic deliberately rejects edge predictions that become too weakly constrained.
- Fine-grained FM recovery is not being developed further in this beta. FM cells are treated by the same support and detector rules as the accepted approach.
- Band and comb triggers act on groups, then flag the participating region/tooth cells in unison for that integration. Point and cross-change tests remain cellwise.
- This notebook does not write products. Use `eigsep-rfi-v3-beta` on complete files after reviewing this result.

## Decision requested

Review whether this updated `v3-beta` result is suitable for broader multi-file testing and product generation. Approval should specify the file range and whether overwriting any existing beta payload is allowed.

**STOPPED AT REVIEW GATE — awaiting Aaron's approval.**